In [1]:
# ============================================================
# IMPORTS
# ============================================================

import numpy as np
import pandas as pd

from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense

from pyswarm import pso

In [2]:
# ============================================================
# LOAD DATA
# ============================================================

df = pd.read_csv("NIFTY_5_Years.csv")

df.columns = df.columns.str.strip()
df['Date'] = pd.to_datetime(df['Date'])

df.set_index('Date', inplace=True)
df = df.sort_index()

data = df[['Close']]

# Train-Test Split
train_size = int(len(data) * 0.8)
train, test = data[:train_size], data[train_size:]

# Scaling
scaler = MinMaxScaler()
train_scaled = scaler.fit_transform(train)
test_scaled = scaler.transform(test)

print("Train:", len(train), "Test:", len(test))

Train: 992 Test: 248


In [3]:
# ============================================================
# SEQUENCE FUNCTION
# ============================================================

def create_sequences(data, time_steps):
    X, y = [], []
    
    for i in range(len(data) - time_steps):
        X.append(data[i:i+time_steps])
        y.append(data[i+time_steps])
    
    return np.array(X), np.array(y)

In [4]:
# ============================================================
# OBJECTIVE FUNCTION
# ============================================================

def objective_function(params):
    try:
        p, d, q, P, D, Q, m, units = map(int, params)
        time_steps = 30

        # SARIMA
        model = SARIMAX(train['Close'],
                        order=(p,d,q),
                        seasonal_order=(P,D,Q,m))

        result = model.fit(disp=False)
        forecast = result.forecast(steps=len(test))

        # Residuals
        residuals = test['Close'].values - forecast.values
        residuals = residuals.reshape(-1,1)

        scaler_res = MinMaxScaler()
        res_scaled = scaler_res.fit_transform(residuals)

        X_res, y_res = create_sequences(res_scaled, time_steps)

        if len(X_res) == 0:
            return 1e10

        # LSTM
        model_lstm = Sequential([
            LSTM(units, input_shape=(time_steps,1)),
            Dense(1)
        ])

        model_lstm.compile(optimizer='adam', loss='mse')
        model_lstm.fit(X_res, y_res, epochs=10, batch_size=32, verbose=0)

        pred = model_lstm.predict(X_res, verbose=0)
        pred = scaler_res.inverse_transform(pred)

        final_pred = forecast[time_steps:].values + pred.flatten()
        actual = test['Close'].values[time_steps:]

        rmse = np.sqrt(mean_squared_error(actual, final_pred))

        return rmse

    except:
        return 1e10

In [5]:
# ============================================================
# PARAMETER BOUNDS
# ============================================================

lb = [1, 0, 1,   0, 0, 0,   5, 32]    # lower bounds
ub = [5, 1, 5,   2, 1, 2,  21, 128]   # upper bounds

In [6]:
# ============================================================
# RUN PSO
# ============================================================

best_params, best_score = pso(
    objective_function,
    lb,
    ub,
    swarmsize=40,
    maxiter=20
)

print("\n===== BEST RESULT =====")
print("Best Parameters:", best_params)
print("Best RMSE:", best_score)

C:\Users\Asus\AppData\Local\Programs\Python\Python310\lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
C:\Users\Asus\AppData\Local\Programs\Python\Python310\lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
C:\Users\Asus\AppData\Local\Programs\Python\Python310\lib\site-packages\statsmodels\tsa\statespace\sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'
C:\Users\Asus\AppData\Local\Programs\Python\Python310\lib\site-packages\statsmodels\tsa\statespace\sarimax.py:997: UserWarning: Non-stationary starting seas

Stopping search: maximum iterations reached --> 20

===== BEST RESULT =====
Best Parameters: [  3.23087068   0.80713284   1.08607434   1.49013403   0.18033362
   0.82482803  12.50028911 116.28140088]
Best RMSE: 10000000000.0


C:\Users\Asus\AppData\Local\Programs\Python\Python310\lib\site-packages\statsmodels\tsa\base\tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
C:\Users\Asus\AppData\Local\Programs\Python\Python310\lib\site-packages\statsmodels\tsa\base\tsa_model.py:837: FutureWarning: No supported index is available. In the next version, calling this method in a model without a supported index will result in an exception.
  return get_prediction_index(


In [7]:
# ============================================================
# TRAIN FINAL MODEL WITH BEST PARAMS
# ============================================================

p, d, q, P, D, Q, m, units = map(int, best_params)

print("Using Best Params:", p, d, q, P, D, Q, m, units)

Using Best Params: 3 0 1 1 0 0 12 116
